In [1]:

required_fields = {
    "olist_customers_dataset.csv": [
        "customer_id",
        "customer_unique_id"
    ],

    "olist_geolocation_dataset.csv": [
        "geolocation_zip_code_prefix",
        "geolocation_lat",
        "geolocation_lng"
    ],

    "olist_order_items_dataset.csv": [
        "order_id",
        "order_item_id",
        "product_id",
        "seller_id",
        "price",
        "freight_value"
    ],

    "olist_order_payments_dataset.csv": [
        "order_id",
        "payment_sequential",
        "payment_type",
        "payment_value"
    ],

    "olist_order_reviews_dataset.csv": [
        "review_id",
        "order_id",
        "review_score"
    ],

    "olist_orders_dataset.csv": [
        "order_id",
        "customer_id",
        "order_status",
        "order_purchase_timestamp"
    ],


    "olist_products_dataset.csv": [
        "product_id"
    ],

    "olist_sellers_dataset.csv": [
        "seller_id"
    ],

    "product_category_name_translation.csv": [
        "product_category_name",
        "product_category_name_english"
    ]
}

for filename, columns in required_fields.items():
    print(f"{filename}: {columns}")


olist_customers_dataset.csv: ['customer_id', 'customer_unique_id']
olist_geolocation_dataset.csv: ['geolocation_zip_code_prefix', 'geolocation_lat', 'geolocation_lng']
olist_order_items_dataset.csv: ['order_id', 'order_item_id', 'product_id', 'seller_id', 'price', 'freight_value']
olist_order_payments_dataset.csv: ['order_id', 'payment_sequential', 'payment_type', 'payment_value']
olist_order_reviews_dataset.csv: ['review_id', 'order_id', 'review_score']
olist_orders_dataset.csv: ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp']
olist_products_dataset.csv: ['product_id']
olist_sellers_dataset.csv: ['seller_id']
product_category_name_translation.csv: ['product_category_name', 'product_category_name_english']


In [ ]:
import csv
from pathlib import Path
data_folder = Path("data")
errors = []
total_rows = 0
for filename, columns in required_fields.items():
    file_path = data_folder / filename
    if not file_path.exists():
        print(f"File not found: {file_path}")
        continue
    with open(file_path, "r", encoding="utf-8-sig", newline="") as file:
        reader = csv.DictReader(file)
        for row_number, row in enumerate(reader, start=2):
            total_rows += 1
            for column in columns:
                value = row.get(column)
                if value is None or str(value).strip() == "":
                    errors.append({
                        "file": filename,
                        "row": row_number,
                        "column": column,
                        "reason": "Missing required value"
                    })
print("Total rows checked:", total_rows)
print("Total errors found:", len(errors))
for error in errors[:10]:
    print(error)


File not found: data\olist_customers_dataset.csv
File not found: data\olist_geolocation_dataset.csv
File not found: data\olist_order_items_dataset.csv
File not found: data\olist_order_payments_dataset.csv
File not found: data\olist_order_reviews_dataset.csv
File not found: data\olist_orders_dataset.csv
File not found: data\olist_products_dataset.csv
File not found: data\olist_sellers_dataset.csv
File not found: data\product_category_name_translation.csv
Total rows checked: 0
Total errors found: 0


In [ ]:
import csv
import json
from collections import defaultdict
errors_by_row = defaultdict(list)
for error in errors:
    key = (error["file"], error["row"])
    errors_by_row[key].append(
        f'{error["column"]}: {error["reason"]}'
    )
bad_records = []
for filename in required_fields:
    file_path = data_folder / filename
    if not file_path.exists():
        continue
    with open(file_path, "r", encoding="utf-8-sig", newline="") as file:
        reader = csv.DictReader(file)
        for row_number, row in enumerate(reader, start=2):
            reasons = errors_by_row.get((filename, row_number))
            if reasons:
                bad_records.append({
                    "file": filename,
                    "row_number": row_number,
                    "reason": "; ".join(reasons),
                    "record": json.dumps(row, ensure_ascii=False)
                })
with open("bad_records.csv", "w", encoding="utf-8", newline="") as file:
    writer = csv.DictWriter(
        file,
        fieldnames=["file", "row_number", "reason", "record"]
    )

    writer.writeheader()
    writer.writerows(bad_records)

print("Rejected rows:", len(bad_records))
print("Saved to bad_records.csv")


Rejected rows: 0
Saved to bad_records.csv


In [4]:

import csv
from decimal import Decimal, InvalidOperation
from datetime import datetime

validation_rules = {
    "olist_order_items_dataset.csv": {
        "numeric": ["price", "freight_value"],
        "dates": ["shipping_limit_date"]
    },
    "olist_order_payments_dataset.csv": {
        "numeric": ["payment_value"],
        "dates": []
    },
    "olist_orders_dataset.csv": {
        "numeric": [],
        "dates": [
            "order_purchase_timestamp",
            "order_approved_at",
            "order_delivered_carrier_date",
            "order_delivered_customer_date",
            "order_estimated_delivery_date"
        ]
    },
    "olist_order_reviews_dataset.csv": {
        "numeric": [],
        "dates": [
            "review_creation_date",
            "review_answer_timestamp"
        ]
    }
}

type_errors = []

for filename, rules in validation_rules.items():

    file_path = data_folder / filename

    if not file_path.exists():
        print(f"File not found: {filename}")
        continue

    with open(file_path, "r", encoding="utf-8-sig", newline="") as file:

        reader = csv.DictReader(file)

        for row_number, row in enumerate(reader, start=2):

            for column in rules["numeric"]:

                value = row.get(column)

                if value is None or str(value).strip() == "":
                    continue

                try:
                    number = Decimal(value)

                    if not number.is_finite():
                        raise InvalidOperation

                except InvalidOperation:
                    type_errors.append({
                        "file": filename,
                        "row": row_number,
                        "column": column,
                        "reason": "Invalid numeric value"
                    })

            for column in rules["dates"]:

                value = row.get(column)

                if value is None or str(value).strip() == "":
                    continue

                try:
                    datetime.fromisoformat(value.strip())

                except ValueError:
                    type_errors.append({
                        "file": filename,
                        "row": row_number,
                        "column": column,
                        "reason": "Invalid date format"
                    })

errors = [
    error for error in errors
    if error["reason"] not in (
        "Invalid numeric value",
        "Invalid date format"
    )
]

errors.extend(type_errors)

print("Task 4 errors found:", len(type_errors))
print("Total collected errors:", len(errors))

for error in type_errors[:10]:
    print(error)


File not found: olist_order_items_dataset.csv
File not found: olist_order_payments_dataset.csv
File not found: olist_orders_dataset.csv
File not found: olist_order_reviews_dataset.csv
Task 4 errors found: 0
Total collected errors: 0


In [5]:

import csv
from decimal import Decimal, InvalidOperation

allowed_statuses = {
    "created",
    "approved",
    "invoiced",
    "processing",
    "shipped",
    "delivered",
    "canceled",
    "unavailable"
}

business_errors = []

items_file = data_folder / "olist_order_items_dataset.csv"

if items_file.exists():

    with open(items_file, "r", encoding="utf-8-sig", newline="") as file:

        reader = csv.DictReader(file)

        for row_number, row in enumerate(reader, start=2):

            price = row.get("price")

            if price is None or price.strip() == "":
                continue

            try:
                price_value = Decimal(price)

                if price_value.is_finite() and price_value <= 0:
                    business_errors.append({
                        "file": items_file.name,
                        "row": row_number,
                        "column": "price",
                        "reason": "Price must be greater than zero"
                    })

            except InvalidOperation:
                pass

else:
    print(f"File not found: {items_file}")


orders_file = data_folder / "olist_orders_dataset.csv"

if orders_file.exists():

    with open(orders_file, "r", encoding="utf-8-sig", newline="") as file:

        reader = csv.DictReader(file)

        for row_number, row in enumerate(reader, start=2):

            status = row.get("order_status")

            if status is None or status.strip() == "":
                continue

            if status.strip().lower() not in allowed_statuses:

                business_errors.append({
                    "file": orders_file.name,
                    "row": row_number,
                    "column": "order_status",
                    "reason": "Invalid order status"
                })

else:
    print(f"File not found: {orders_file}")


for error in business_errors:
    if error not in errors:
        errors.append(error)

print("Business rule errors:", len(business_errors))
print("Total collected errors:", len(errors))

for error in business_errors[:10]:
    print(error)


File not found: data\olist_order_items_dataset.csv
File not found: data\olist_orders_dataset.csv
Business rule errors: 0
Total collected errors: 0


In [6]:

import csv
import re

zip_pattern = re.compile(r"^[0-9]{5}$")
id_pattern = re.compile(r"^[0-9a-fA-F]{32}$")

zip_fields = {
    "olist_customers_dataset.csv": ["customer_zip_code_prefix"],
    "olist_sellers_dataset.csv": ["seller_zip_code_prefix"],
    "olist_geolocation_dataset.csv": ["geolocation_zip_code_prefix"]
}

id_fields = {
    "customer_id", "customer_unique_id",
    "order_id", "product_id", "seller_id", "review_id"
}

regex_errors = []

for filename in required_fields:

    file_path = data_folder / filename

    if not file_path.exists():
        continue

    with open(file_path, "r", encoding="utf-8-sig", newline="") as file:

        reader = csv.DictReader(file)

        for row_number, row in enumerate(reader, start=2):

            for column in zip_fields.get(filename, []):

                if column in row:
                    value = (row.get(column) or "").strip()

                    if not zip_pattern.fullmatch(value):
                        regex_errors.append({
                            "file": filename,
                            "row": row_number,
                            "column": column,
                            "reason": "ZIP must contain exactly 5 digits"
                        })

            for column in id_fields.intersection(row.keys()):

                value = (row.get(column) or "").strip()

                if value and not id_pattern.fullmatch(value):
                    regex_errors.append({
                        "file": filename,
                        "row": row_number,
                        "column": column,
                        "reason": "ID must contain 32 hexadecimal characters"
                    })

for error in regex_errors:
    if error not in errors:
        errors.append(error)

print("Regex errors:", len(regex_errors))

for error in regex_errors[:10]:
    print(error)


Regex errors: 0


In [7]:
class SchemaError(Exception):
    pass

class FileFormatError(Exception):
    pass

print("Custom exceptions created successfully")



Custom exceptions created successfully


In [15]:

import csv

file_issues = {}

for filename, required_columns in required_fields.items():

    file_path = data_folder / filename

    try:
        if not file_path.is_file():
            raise FileFormatError("File not found")

        if file_path.stat().st_size == 0:
            raise FileFormatError("File is empty")

        with open(file_path, "r", encoding="utf-8-sig", newline="") as file:

            reader = csv.DictReader(file)

            if not reader.fieldnames:
                raise FileFormatError("CSV header is missing")

            if len(reader.fieldnames) != len(set(reader.fieldnames)):
                raise SchemaError("Duplicate column names")

            missing_columns = [
                column for column in required_columns
                if column not in reader.fieldnames
            ]

            if missing_columns:
                raise SchemaError(
                    f"Missing columns: {missing_columns}"
                )

            if next(reader, None) is None:
                raise FileFormatError("CSV has no data rows")

        print(f"PASS: {filename}")

    except (SchemaError, FileFormatError, OSError,
            UnicodeError, csv.Error) as e:

        file_issues[filename] = str(e)
        print(f"FAIL: {filename} - {e}")

    finally:
        print(f"Finished checking: {filename}")

print("Files with issues:", len(file_issues))


PASS: olist_customers_dataset.csv
Finished checking: olist_customers_dataset.csv
PASS: olist_geolocation_dataset.csv
Finished checking: olist_geolocation_dataset.csv
PASS: olist_order_items_dataset.csv
Finished checking: olist_order_items_dataset.csv
PASS: olist_order_payments_dataset.csv
Finished checking: olist_order_payments_dataset.csv
PASS: olist_order_reviews_dataset.csv
Finished checking: olist_order_reviews_dataset.csv
PASS: olist_orders_dataset.csv
Finished checking: olist_orders_dataset.csv
PASS: olist_products_dataset.csv
Finished checking: olist_products_dataset.csv
PASS: olist_sellers_dataset.csv
Finished checking: olist_sellers_dataset.csv
PASS: product_category_name_translation.csv
Finished checking: product_category_name_translation.csv
Files with issues: 0


In [9]:

from pathlib import Path

print("Current folder:", Path.cwd())
print("Data folder:", data_folder.resolve())

csv_files = list(Path.cwd().rglob("*.csv"))

print("CSV files found:", len(csv_files))

for file in csv_files[:20]:
    print(file)


Current folder: c:\tap-python-git
Data folder: C:\tap-python-git\data
CSV files found: 1
c:\tap-python-git\bad_records.csv


In [10]:
data_folder = Path("data")

In [12]:

from pathlib import Path

data_root = Path(r"C:\tap-python-git\data")

csv_files = list(data_root.rglob("*.csv"))

print("CSV files found:", len(csv_files))

for file in csv_files:
    print(file)

missing = [
    name for name in required_fields
    if not any(p.name == name for p in csv_files)
]

print("\nMissing required CSV files:", missing)

if not missing:
    folders = {
        p.parent for p in csv_files
        if p.name in required_fields
    }

    if len(folders) == 1:
        data_folder = folders.pop()
        print("Data folder configured:", data_folder)
    else:
        print("Required CSV files are in multiple folders")


CSV files found: 9
C:\tap-python-git\data\customers.csv
C:\tap-python-git\data\geolocation.csv
C:\tap-python-git\data\orders.csv
C:\tap-python-git\data\order_items.csv
C:\tap-python-git\data\order_payments.csv
C:\tap-python-git\data\order_reviews.csv
C:\tap-python-git\data\products.csv
C:\tap-python-git\data\product_category_name_translation.csv
C:\tap-python-git\data\sellers.csv

Missing required CSV files: ['olist_customers_dataset.csv', 'olist_geolocation_dataset.csv', 'olist_order_items_dataset.csv', 'olist_order_payments_dataset.csv', 'olist_order_reviews_dataset.csv', 'olist_orders_dataset.csv', 'olist_products_dataset.csv', 'olist_sellers_dataset.csv']


In [13]:


$folder = "C:\tap-python-git\data"

$files = @{
    "customers.csv" = "olist_customers_dataset.csv"
    "geolocation.csv" = "olist_geolocation_dataset.csv"
    "orders.csv" = "olist_orders_dataset.csv"
    "order_items.csv" = "olist_order_items_dataset.csv"
    "order_payments.csv" = "olist_order_payments_dataset.csv"
    "order_reviews.csv" = "olist_order_reviews_dataset.csv"
    "products.csv" = "olist_products_dataset.csv"
    "sellers.csv" = "olist_sellers_dataset.csv"
}

foreach ($old in $files.Keys) {
    Copy-Item "$folder\$old" "$folder\$($files[$old])"
}

Write-Host "CSV filenames ready!"


SyntaxError: invalid syntax (3290362504.py, line 1)

In [14]:

from pathlib import Path
import shutil

data_folder = Path(r"C:\tap-python-git\data")

file_names = {
    "customers.csv": "olist_customers_dataset.csv",
    "geolocation.csv": "olist_geolocation_dataset.csv",
    "orders.csv": "olist_orders_dataset.csv",
    "order_items.csv": "olist_order_items_dataset.csv",
    "order_payments.csv": "olist_order_payments_dataset.csv",
    "order_reviews.csv": "olist_order_reviews_dataset.csv",
    "products.csv": "olist_products_dataset.csv",
    "sellers.csv": "olist_sellers_dataset.csv"
}

for old_name, new_name in file_names.items():
    source = data_folder / old_name
    destination = data_folder / new_name

    if source.exists():
        if not destination.exists():
            shutil.copy2(source, destination)
        print(f"Ready: {new_name}")
    else:
        print(f"Missing: {old_name}")

missing = [
    filename for filename in required_fields
    if not (data_folder / filename).exists()
]

print("\nMissing files:", missing)
print("Files ready:", len(required_fields) - len(missing))


Ready: olist_customers_dataset.csv
Ready: olist_geolocation_dataset.csv
Ready: olist_orders_dataset.csv
Ready: olist_order_items_dataset.csv
Ready: olist_order_payments_dataset.csv
Ready: olist_order_reviews_dataset.csv
Ready: olist_products_dataset.csv
Ready: olist_sellers_dataset.csv

Missing files: []
Files ready: 9


In [16]:

import csv

file_issues = {}

for filename, required_columns in required_fields.items():

    file_path = data_folder / filename

    try:
        if not file_path.is_file():
            raise FileFormatError("File not found")

        if file_path.stat().st_size == 0:
            raise FileFormatError("File is empty")

        with open(file_path, "r", encoding="utf-8-sig", newline="") as file:

            reader = csv.DictReader(file)

            if not reader.fieldnames:
                raise FileFormatError("CSV header is missing")

            if len(reader.fieldnames) != len(set(reader.fieldnames)):
                raise SchemaError("Duplicate column names")

            missing_columns = [
                column for column in required_columns
                if column not in reader.fieldnames
            ]

            if missing_columns:
                raise SchemaError(
                    f"Missing columns: {missing_columns}"
                )

            if next(reader, None) is None:
                raise FileFormatError("CSV has no data rows")

        print(f"PASS: {filename}")

    except (SchemaError, FileFormatError, OSError,
            UnicodeError, csv.Error) as e:

        file_issues[filename] = str(e)
        print(f"FAIL: {filename} - {e}")

    finally:
        print(f"Finished checking: {filename}")

print("\nFiles with issues:", len(file_issues))


PASS: olist_customers_dataset.csv
Finished checking: olist_customers_dataset.csv
PASS: olist_geolocation_dataset.csv
Finished checking: olist_geolocation_dataset.csv
PASS: olist_order_items_dataset.csv
Finished checking: olist_order_items_dataset.csv
PASS: olist_order_payments_dataset.csv
Finished checking: olist_order_payments_dataset.csv
PASS: olist_order_reviews_dataset.csv
Finished checking: olist_order_reviews_dataset.csv
PASS: olist_orders_dataset.csv
Finished checking: olist_orders_dataset.csv
PASS: olist_products_dataset.csv
Finished checking: olist_products_dataset.csv
PASS: olist_sellers_dataset.csv
Finished checking: olist_sellers_dataset.csv
PASS: product_category_name_translation.csv
Finished checking: product_category_name_translation.csv

Files with issues: 0


In [17]:

import csv

summary = []

for filename in required_fields:

    file_path = data_folder / filename

    total = 0

    if file_path.is_file():

        try:
            with open(file_path, "r", encoding="utf-8-sig", newline="") as file:

                reader = csv.DictReader(file)

                total = sum(1 for row in reader)

        except (OSError, UnicodeError, csv.Error) as e:
            file_issues[filename] = str(e)

    rejected_rows = {
        error["row"]
        for error in errors
        if error["file"] == filename
    }

    if filename in file_issues:
        valid = 0
        rejected = total
        status = "FAIL"

    else:
        rejected = len(rejected_rows)
        valid = total - rejected
        status = "PASS"

    summary.append({
        "file": filename,
        "total": total,
        "valid": valid,
        "rejected": rejected,
        "status": status
    })

print("DAY 13 VALIDATION SUMMARY")
print("-" * 60)

for result in summary:
    print(
        f"{result['file']} | "
        f"Total: {result['total']} | "
        f"Valid: {result['valid']} | "
        f"Rejected: {result['rejected']} | "
        f"{result['status']}"
    )

print("-" * 60)

print("Total rows:", sum(x["total"] for x in summary))
print("Valid rows:", sum(x["valid"] for x in summary))
print("Rejected rows:", sum(x["rejected"] for x in summary))


DAY 13 VALIDATION SUMMARY
------------------------------------------------------------
olist_customers_dataset.csv | Total: 500 | Valid: 500 | Rejected: 0 | PASS
olist_geolocation_dataset.csv | Total: 2229 | Valid: 2229 | Rejected: 0 | PASS
olist_order_items_dataset.csv | Total: 576 | Valid: 576 | Rejected: 0 | PASS
olist_order_payments_dataset.csv | Total: 545 | Valid: 545 | Rejected: 0 | PASS
olist_order_reviews_dataset.csv | Total: 489 | Valid: 489 | Rejected: 0 | PASS
olist_orders_dataset.csv | Total: 500 | Valid: 500 | Rejected: 0 | PASS
olist_products_dataset.csv | Total: 512 | Valid: 512 | Rejected: 0 | PASS
olist_sellers_dataset.csv | Total: 317 | Valid: 317 | Rejected: 0 | PASS
product_category_name_translation.csv | Total: 71 | Valid: 71 | Rejected: 0 | PASS
------------------------------------------------------------
Total rows: 5739
Valid rows: 5739
Rejected rows: 0


In [18]:

MAX_REJECTION_RATE = 0.20

for result in summary:

    filename = result["file"]
    total = result["total"]
    rejected = result["rejected"]

    try:
        if result["status"] == "FAIL":
            raise FileFormatError(
                file_issues.get(filename, "File validation failed")
            )

        if total == 0:
            raise FileFormatError("No records available")

        rejection_rate = rejected / total

        if rejection_rate > MAX_REJECTION_RATE:
            raise FileFormatError(
                f"Rejection rate {rejection_rate:.2%} exceeds 20%"
            )

        print(f"PASS: {filename} - {rejection_rate:.2%}")

    except FileFormatError as e:
        result["status"] = "FAIL"
        print(f"FAIL: {filename} - {e}")

print("\nFiles passed:", sum(x["status"] == "PASS" for x in summary))
print("Files failed:", sum(x["status"] == "FAIL" for x in summary))


PASS: olist_customers_dataset.csv - 0.00%
PASS: olist_geolocation_dataset.csv - 0.00%
PASS: olist_order_items_dataset.csv - 0.00%
PASS: olist_order_payments_dataset.csv - 0.00%
PASS: olist_order_reviews_dataset.csv - 0.00%
PASS: olist_orders_dataset.csv - 0.00%
PASS: olist_products_dataset.csv - 0.00%
PASS: olist_sellers_dataset.csv - 0.00%
PASS: product_category_name_translation.csv - 0.00%

Files passed: 9
Files failed: 0
